# Purpose

Play around with code to develop a method of reading raw spectrum data and writing to a differently formatted output file.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from pathlib import Path

## Get raw data files

In [2]:
raw_data = Path('raw_data')
data_files = sorted([f for f in raw_data.glob("*.txt")])
data_files

[PosixPath('raw_data/FLMS155811_13-39-15-538.txt'),
 PosixPath('raw_data/FLMS155811_13-39-28-747.txt'),
 PosixPath('raw_data/FLMS155811_13-43-46-130.txt')]

## Code to read spectrum data

In [3]:
def read_Fire_spectrum_data_file(file, skiprows=14, clip_to_zero=True):
    print(file)
    temp = np.loadtxt(file, skiprows=skiprows)
    if clip_to_zero: temp = np.clip(temp, 0, None)
    header_lines = []
    with file.open() as f:
        for i in range(skiprows):
            header_lines.append(f.readline().strip())
    return header_lines, temp

## Read data and header lines

In [4]:
header, data = read_Fire_spectrum_data_file(data_files[0])
header

raw_data/FLMS155811_13-39-15-538.txt


['Data from FLMS155811_13-39-15-538.txt Node',
 '',
 'Date: Tue Dec 31 13:39:15 MST 2019',
 'User: ECEn Student',
 'Spectrometer: FLMS15581',
 'Trigger mode: 0',
 'Integration Time (sec): 1.000000E-1',
 'Scans to average: 1',
 'Electric dark correction enabled: true',
 'Nonlinearity correction enabled: false',
 'Boxcar width: 0',
 'XAxis mode: Wavelengths',
 'Number of Pixels in Spectrum: 2048',
 '>>>>>Begin Spectral Data<<<<<']

## Try creating new header string

In [5]:
new_header = []
for s in header:
    new_header.append(" ".join(("#", s)))
new_header

['# Data from FLMS155811_13-39-15-538.txt Node',
 '# ',
 '# Date: Tue Dec 31 13:39:15 MST 2019',
 '# User: ECEn Student',
 '# Spectrometer: FLMS15581',
 '# Trigger mode: 0',
 '# Integration Time (sec): 1.000000E-1',
 '# Scans to average: 1',
 '# Electric dark correction enabled: true',
 '# Nonlinearity correction enabled: false',
 '# Boxcar width: 0',
 '# XAxis mode: Wavelengths',
 '# Number of Pixels in Spectrum: 2048',
 '# >>>>>Begin Spectral Data<<<<<']

In [6]:
new_header = ['# ' + s for s in header if s != '']
new_header

['# Data from FLMS155811_13-39-15-538.txt Node',
 '# Date: Tue Dec 31 13:39:15 MST 2019',
 '# User: ECEn Student',
 '# Spectrometer: FLMS15581',
 '# Trigger mode: 0',
 '# Integration Time (sec): 1.000000E-1',
 '# Scans to average: 1',
 '# Electric dark correction enabled: true',
 '# Nonlinearity correction enabled: false',
 '# Boxcar width: 0',
 '# XAxis mode: Wavelengths',
 '# Number of Pixels in Spectrum: 2048',
 '# >>>>>Begin Spectral Data<<<<<']

In [7]:
new_header_str = "\n".join(new_header)
new_header_str

'# Data from FLMS155811_13-39-15-538.txt Node\n# Date: Tue Dec 31 13:39:15 MST 2019\n# User: ECEn Student\n# Spectrometer: FLMS15581\n# Trigger mode: 0\n# Integration Time (sec): 1.000000E-1\n# Scans to average: 1\n# Electric dark correction enabled: true\n# Nonlinearity correction enabled: false\n# Boxcar width: 0\n# XAxis mode: Wavelengths\n# Number of Pixels in Spectrum: 2048\n# >>>>>Begin Spectral Data<<<<<'

In [8]:
new_header_str = "\n".join(['# ' + s for s in header if s != ''])
new_header_str

'# Data from FLMS155811_13-39-15-538.txt Node\n# Date: Tue Dec 31 13:39:15 MST 2019\n# User: ECEn Student\n# Spectrometer: FLMS15581\n# Trigger mode: 0\n# Integration Time (sec): 1.000000E-1\n# Scans to average: 1\n# Electric dark correction enabled: true\n# Nonlinearity correction enabled: false\n# Boxcar width: 0\n# XAxis mode: Wavelengths\n# Number of Pixels in Spectrum: 2048\n# >>>>>Begin Spectral Data<<<<<'

## Actually, you can use `header` directly

In [9]:
data_file = Path('measured_spectrum.csv')
np.savetxt(data_file, data, delimiter=',', header="\n".join(header))

## Read all data files...

In [10]:
all_data = []
all_headers = []
for f in data_files:
    temp_header, temp_data = read_Fire_spectrum_data_file(f)
    all_data.append(temp_data)
    all_headers.append(temp_header)

raw_data/FLMS155811_13-39-15-538.txt
raw_data/FLMS155811_13-39-28-747.txt
raw_data/FLMS155811_13-43-46-130.txt


In [11]:
all_headers[0][0][10:37]

'FLMS155811_13-39-15-538.txt'

## ...and plot

In [12]:
fig, ax = plt.subplots()
for h,d in zip(all_headers, all_data):
    ax.plot(d[:,0], d[:,1], label=h[0][10:37])
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Power (ADC counts)')
ax.set_title('With 370 nm Short Pass Filter')
ax.grid()
ax.legend();

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [13]:
data = all_data[0]

data_normalized = data.copy()
data_normalized[:, 1] = data_normalized[:, 1] / np.amax(data_normalized[:, 1])
data_normalized

array([[2.98026000e+02, 0.00000000e+00],
       [2.98106000e+02, 0.00000000e+00],
       [2.98185000e+02, 0.00000000e+00],
       ...,
       [4.34114000e+02, 6.98319005e-03],
       [4.34167000e+02, 6.98319005e-03],
       [4.34221000e+02, 6.98319005e-03]])

In [14]:
np.amax(data_normalized[:, 1])

1.0

In [15]:
max_index = np.argmax(data[:,1])
max_value = np.amax(data[:,1])

print(max_index, max_value, data[max_index])

961 57621.23 [  368.573 57621.23 ]


In [17]:
data_files[0]

PosixPath('raw_data/FLMS155811_13-39-15-538.txt')